<a href="https://colab.research.google.com/github/doomchitdoomchit/OD_DAY_SIDO_SEOULGU/blob/master/notebook/%EC%A7%80%EC%97%AD%EB%B3%84_%EC%83%81%ED%92%88_%EB%AC%BC%EB%8F%99%EB%9F%89_%ED%86%B5%EA%B3%84_%EB%8D%B0%EC%9D%B4%ED%84%B0_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os

import polars as pl
import numpy as np
import lightgbm as lgb

from sklearn.metrics import mean_squared_error, mean_absolute_error

In [34]:
drive_path = '/content/drive/My Drive/Colab Notebooks/지역별 상품 물동량 통계 데이터'
data_name = 'seoulgu.parquet'

In [13]:
# data load
lazy_df = pl.scan_parquet(os.path.join(drive_path, data_name))

In [14]:
all_cols = lazy_df.columns
cat_cols = [c for c in all_cols if c.startswith('CAT_VOL_')]

/tmp/ipykernel_3118/2883817882.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  all_cols = lazy_df.columns


In [21]:
# 날짜 타입 변환 및 11개 카테고리 수평 합산 (total_vol)
lazy_processed = lazy_df.with_columns([
    pl.col('DELIVERY_YMD').cast(pl.Date).alias('DELIVERY_YMD'),
    pl.sum_horizontal(cat_cols).alias('total_vol') ])


In [22]:
# (2) 자치구 및 일자별 집계 및 정렬
lazy_daily = (
    lazy_processed.group_by(['DELIVERY_YMD', 'RECEIVER_SIGUNGU_NM']).agg(
        pl.col('total_vol').sum()
        ).sort(['RECEIVER_SIGUNGU_NM', 'DELIVERY_YMD'])
    )


In [23]:
# (3) 시계열 피처 생성 (캘린더 + 시차 Lag + 이동 통계량 Rolling)
lags = [7, 14, 21, 28]
windows = [7, 14, 28]

# Lag 피처 표현식
lag_exprs = [
    pl.col('total_vol').shift(lag).over('RECEIVER_SIGUNGU_NM').alias(f'vol_lag_{lag}')
    for lag in lags
]

# Rolling 피처 표현식 (Data Leakage 방지를 위해 Shift 7 적용)
rolling_exprs = []
for w in windows:
    rolling_exprs.append(
        pl.col('total_vol').shift(7).rolling_mean(window_size=w)
        .over('RECEIVER_SIGUNGU_NM').alias(f'rolling_mean_{w}'))
    rolling_exprs.append(
        pl.col('total_vol').shift(7).rolling_std(window_size=w)
        .over('RECEIVER_SIGUNGU_NM').alias(f'rolling_std_{w}'))


In [24]:
# 지연 피처 엔지니어링 쿼리 결합
lazy_featured = lazy_daily.with_columns([
    pl.col('DELIVERY_YMD').dt.year().alias('year'),
    pl.col('DELIVERY_YMD').dt.month().alias('month'),
    pl.col('DELIVERY_YMD').dt.day().alias('day'),
    pl.col('DELIVERY_YMD').dt.weekday().alias('dayofweek'),
    pl.col('DELIVERY_YMD').dt.weekday().is_in([6, 7]).cast(pl.Int8).alias('is_weekend'),
    pl.col('RECEIVER_SIGUNGU_NM').cast(pl.Categorical)
    ] + lag_exprs + rolling_exprs).drop_nulls()

In [25]:
print("Lazy Query Plan 최적화 및 연산 실행 중...")
df_processed = lazy_featured.collect()
print(f"연산 완료된 데이터 Shape: {df_processed.shape}")

Lazy Query Plan 최적화 및 연산 실행 중...
연산 완료된 데이터 Shape: (44779, 18)


In [27]:
max_date = df_processed['DELIVERY_YMD'].max()
split_date = max_date - pl.duration(days=90)

# 최근 3개월 검증
train_df = df_processed.filter(pl.col('DELIVERY_YMD') < split_date)
test_df = df_processed.filter(pl.col('DELIVERY_YMD') >= split_date)
features = [ 'RECEIVER_SIGUNGU_NM', 'month', 'day', 'dayofweek', 'is_weekend',
            'vol_lag_7', 'vol_lag_14', 'vol_lag_21', 'vol_lag_28',
             'rolling_mean_7', 'rolling_std_7',
             'rolling_mean_14', 'rolling_std_14',
             'rolling_mean_28', 'rolling_std_28'
]
target = 'total_vol'

# LightGBM 입력을 위한 Pandas 변환 (Arrow 메모리 0-Copy 기반 변환)
X_train = train_df.select(features).to_pandas()
y_train = train_df.select(target).to_pandas().squeeze()
X_test = test_df.select(features).to_pandas()
y_test = test_df.select(target).to_pandas().squeeze()

In [28]:
model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=['RECEIVER_SIGUNGU_NM'],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

preds = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
wape = (np.sum(np.abs(y_test - preds)) / np.sum(y_test)) * 100

print(f"\n=== LazyFrame 처리 기반 LightGBM 최종 백테스팅 결과 ===")
print(f"검증 기간: {split_date} ~ {max_date}")
print(f"RMSE : {rmse:,.2f} 건")
print(f"MAE  : {mae:,.2f} 건")
print(f"MAPE : {mape:.2f} %")
print(f"WAPE : {wape:.2f} %")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007946 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2631
[LightGBM] [Info] Number of data points in the train set: 42504, number of used features: 15
[LightGBM] [Info] Start training from score 35640.894410

=== LazyFrame 처리 기반 LightGBM 최종 백테스팅 결과 ===
검증 기간: [(2023-12-31) - (90d.alias("duration"))] ~ 2023-12-31
RMSE : 11,355.02 건
MAE  : 5,784.53 건
MAPE : 61.28 %
WAPE : 16.33 %


pandas.core.frame.DataFrame